In [ ]:
# Example 1: FifoMemoryQueue - in-memory first-in, first-out queue.
# The memory queues live in queuelib.queue; the top-level __init__ only
# re-exports the disk queues + PriorityQueue/RoundRobinQueue.
from queuelib.queue import FifoMemoryQueue

q = FifoMemoryQueue()
for item in [b"alpha", b"beta", b"gamma"]:
    q.push(item)

print(f"size after pushes: {len(q)}")

first = q.pop()
second = q.pop()
print(f"first popped: {first!r}")
print(f"second popped: {second!r}")

assert first == b"alpha", f"FIFO order broken: expected b'alpha', got {first!r}"
assert second == b"beta"
assert len(q) == 1
q.close()


In [ ]:
# Example 2: LifoMemoryQueue - in-memory last-in, first-out (stack) queue.
from queuelib.queue import LifoMemoryQueue

q = LifoMemoryQueue()
for item in [b"one", b"two", b"three"]:
    q.push(item)

print(f"size after pushes: {len(q)}")

top = q.pop()
next_ = q.pop()
print(f"first popped (top of stack): {top!r}")
print(f"second popped: {next_!r}")

assert top == b"three", f"LIFO order broken: expected b'three', got {top!r}"
assert next_ == b"two"
q.close()


In [ ]:
# Example 3: FifoDiskQueue - persistent on-disk FIFO queue with chunked files.
import tempfile
from queuelib import FifoDiskQueue

with tempfile.TemporaryDirectory() as tmp:
    path = f"{tmp}/fifo"
    q = FifoDiskQueue(path)
    for item in [b"a", b"b", b"c"]:
        q.push(item)
    q.close()

    # Reopen to verify persistence across instances.
    q2 = FifoDiskQueue(path)
    print(f"reopened size: {len(q2)}")
    popped = [q2.pop() for _ in range(3)]
    print(f"popped (FIFO order): {popped}")
    q2.close()

    assert popped == [b"a", b"b", b"c"], f"persistence/order broken: {popped}"


In [ ]:
# Example 4: LifoDiskQueue - persistent on-disk LIFO (stack) queue.
import tempfile
from queuelib import LifoDiskQueue

with tempfile.TemporaryDirectory() as tmp:
    path = f"{tmp}/lifo.db"
    q = LifoDiskQueue(path)
    for item in [b"x", b"y", b"z"]:
        q.push(item)
    print(f"size before close: {len(q)}")
    q.close()

    q2 = LifoDiskQueue(path)
    popped = [q2.pop() for _ in range(3)]
    print(f"popped (LIFO order): {popped}")
    q2.close()

    assert popped == [b"z", b"y", b"x"], f"LIFO disk order broken: {popped}"


In [ ]:
# Example 5: PriorityQueue - dispatches to sub-queues keyed by integer priority.
from queuelib import PriorityQueue
from queuelib.queue import FifoMemoryQueue

def qfactory(priority):
    return FifoMemoryQueue()

pq = PriorityQueue(qfactory)
pq.push(b"low-1", priority=10)
pq.push(b"high-1", priority=1)
pq.push(b"mid-1", priority=5)
pq.push(b"high-2", priority=1)

print(f"total size: {len(pq)}")

order = [pq.pop() for _ in range(4)]
print(f"pop order (lower priority first): {order}")
pq.close()

assert order[0] in (b"high-1", b"high-2"), f"highest-priority not first: {order}"
assert order[-1] == b"low-1", f"lowest-priority not last: {order}"
